In [1]:
import networkit as nk
import pandas as pd
import time
from pathlib import Path

# -- Change these two values before running ------------------------------------
SNAPSHOT = "2026-06-26"
QUERY    = "quantum_comp"
# ------------------------------------------------------------------------------

# Derive input and output paths from SNAPSHOT / QUERY
graph_file  = Path("input")  / f"snapshot_{SNAPSHOT}" / QUERY / "edges.txt"
output_dir  = Path("output") / f"snapshot_{SNAPSHOT}" / QUERY
output_file = output_dir / "louvain_clusters.txt"

# Create output directory if it doesn't exist
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Snapshot: {SNAPSHOT}")
print(f"Query   : {QUERY}")
print(f"Input   : {graph_file}")
print(f"Output  : {output_file}")

# --- 1. Setup ---
print("\nStarting clustering process...")

# Set the number of threads for NetworKit to use.
# Match this with the number of vCPUs on your machine.
nk.setNumberOfThreads(60)
print(f"NetworKit is configured to use {nk.getMaxNumberOfThreads()} threads.")

Snapshot: 2026-06-26
Query   : quantum_comp
Input   : input/snapshot_2026-06-26/quantum_comp/edges.txt
Output  : output/snapshot_2026-06-26/quantum_comp/louvain_clusters.txt

Starting clustering process...
NetworKit is configured to use 60 threads.


In [2]:
# --- 2. Load the Full Graph ---
print(f"\nStep 1/6: Loading full graph from '{graph_file}'...")
start_time = time.time()

# edges.txt is produced by step 03 and downloaded by download_input.ipynb.
G = nk.graphio.readGraph(str(graph_file), fileformat=nk.Format.EdgeListTabZero,
                         directed=False)

end_time = time.time()
print(f"-> Full graph loaded in {end_time - start_time:.2f} seconds.")
print(f"-> Graph has {G.numberOfNodes():,} nodes and {G.numberOfEdges():,} edges.")


Step 1/6: Loading full graph from 'input/snapshot_2026-06-26/quantum_comp/edges.txt'...
-> Full graph loaded in 0.05 seconds.
-> Graph has 32,402 nodes and 285,539 edges.


In [3]:
# --- 3. Find and Extract the Largest Connected Component ---
print("\nStep 2/6: Finding the largest connected component...")
start_time = time.time()

# Calculate connected components
cc = nk.components.ConnectedComponents(G)
cc.run()
component_partition = cc.getPartition()

# Identify the largest component by finding the partition subset with the most members
largest_component_id = max(component_partition.subsetSizeMap(), key=component_partition.subsetSizeMap().get)
largest_component_nodes = component_partition.getMembers(largest_component_id)

end_time = time.time()
print(f"-> Component analysis finished in {end_time - start_time:.2f} seconds.")




Step 2/6: Finding the largest connected component...
-> Component analysis finished in 0.01 seconds.


In [4]:
print("\nStep 3/6: Creating subgraph of the largest component...")
start_time = time.time()
# Create a subgraph containing only the nodes of the largest component.
# This function crucially PRESERVES the original node IDs.
G_lcc = nk.graphtools.subgraphFromNodes(G, nodes=largest_component_nodes, compact=False)

end_time = time.time()
print(f"-> Subgraph created for the largest component.")
print(f"-> The largest component contains {G_lcc.numberOfNodes():,} nodes and {G_lcc.numberOfEdges():,} edges.")
print(f"-> Subgraph created in {end_time - start_time:.2f} seconds.")



Step 3/6: Creating subgraph of the largest component...
-> Subgraph created for the largest component.
-> The largest component contains 31,881 nodes and 285,185 edges.
-> Subgraph created in 0.02 seconds.


---

## 🔴 Micro-level

In [5]:
# --- 4. Run Clustering on the Largest Component ---
print("\nStep 4/6: Running Parallel Louvain Method (PLM) on the largest component...")
start_time = time.time()

# For large graphs >10 million nodes:
micro_plm = nk.community.PLM(G_lcc, refine=True, gamma=10, par='balanced', maxIter=50)
# For networks with <1 million and >100k nodes, try a smaller gamma, e.g. 70.
# micro_plm = nk.community.PLM(G_lcc, refine=True, gamma=70, par='balanced', maxIter=50)

micro_plm.run()
micro_partition = micro_plm.getPartition()

# IMPORTANT: compact subset IDs before building higher-level community graphs.
# This prevents sparse micro IDs from breaking micro->meso mapping later.
micro_partition.compact()

end_time = time.time()
modularity = nk.community.Modularity().getQuality(micro_partition, G_lcc)

print(f"-> Clustering completed in {end_time - start_time:.2f} seconds.")
print(f"-> Found {micro_partition.numberOfSubsets():,} communities with a modularity of {modularity:.4f}.")
print("-> Micro partition IDs were compacted to a dense range [0..K-1].")


Step 4/6: Running Parallel Louvain Method (PLM) on the largest component...
-> Clustering completed in 0.06 seconds.
-> Found 157 communities with a modularity of 0.4364.
-> Micro partition IDs were compacted to a dense range [0..K-1].


In [6]:
# --- 6. Analyze and Display Top Cluster Sizes ---
print("\nStep 6/6: Analyzing sizes of the largest communities...")
start_time = time.time()

# Get a map of {community_id: size}
community_sizes_map = micro_partition.subsetSizeMap()

# Get a list of all community sizes
all_sizes = list(community_sizes_map.values())

# Sort the sizes in descending order
all_sizes.sort()

# --- Display Top 20 Largest ---
# Get the last 20 elements from the sorted list
top_20_sizes = all_sizes[-20:]
# Reverse the new list to display from largest to smallest
top_20_sizes.reverse()

print("\n--- Top 20 Largest Communities ---")
for i, size in enumerate(top_20_sizes, 1):
    print(f"Rank {i:<2}: {size:,} members")




Step 6/6: Analyzing sizes of the largest communities...

--- Top 20 Largest Communities ---
Rank 1 : 1,388 members
Rank 2 : 989 members
Rank 3 : 643 members
Rank 4 : 559 members
Rank 5 : 539 members
Rank 6 : 501 members
Rank 7 : 483 members
Rank 8 : 477 members
Rank 9 : 468 members
Rank 10: 427 members
Rank 11: 420 members
Rank 12: 409 members
Rank 13: 386 members
Rank 14: 376 members
Rank 15: 373 members
Rank 16: 367 members
Rank 17: 365 members
Rank 18: 344 members
Rank 19: 333 members
Rank 20: 320 members


In [7]:
# --- Display Bottom 20 Smallest ---
# Get the first 20 elements from the sorted list
bottom_20_sizes = all_sizes[:20]

print("\n--- Bottom 20 Smallest Communities ---")
for i, size in enumerate(bottom_20_sizes, 1):
    print(f"Rank {i:<2}: {size:,} members")


end_time = time.time()
print(f"-> Successfully summarized cluster sizes in {end_time - start_time:.2f} seconds.")

print("\nProcess complete. ✨")


--- Bottom 20 Smallest Communities ---
Rank 1 : 4 members
Rank 2 : 4 members
Rank 3 : 8 members
Rank 4 : 10 members
Rank 5 : 41 members
Rank 6 : 45 members
Rank 7 : 60 members
Rank 8 : 61 members
Rank 9 : 61 members
Rank 10: 73 members
Rank 11: 74 members
Rank 12: 76 members
Rank 13: 78 members
Rank 14: 81 members
Rank 15: 82 members
Rank 16: 84 members
Rank 17: 84 members
Rank 18: 88 members
Rank 19: 89 members
Rank 20: 90 members
-> Successfully summarized cluster sizes in 0.02 seconds.

Process complete. ✨


In [8]:
# --- Display Number of clusters more than x ---
thr_cls = [num for num in all_sizes if num >= 50]
thr_cnt = sum(thr_cls)
print(f"\nNumber of clusters with more than 50 members: {len(thr_cls):,} | {len(thr_cls) / len(all_sizes) * 100:.2f}% of all clusters")
print(f"They cover {thr_cnt / sum(all_sizes) * 100:.2f}% of all members in the largest component.")


Number of clusters with more than 50 members: 151 | 96.18% of all clusters
They cover 99.65% of all members in the largest component.


In [9]:
# --- 5. Write Results to File ---
print(f"\nStep 5/6: Writing community assignments to '{output_file}'...")
start_time = time.time()

count = 0
with open(output_file, 'w') as f:
    # We iterate through the nodes of the largest component subgraph.
    # The partition object `micro_partition` uses the original node IDs.
    for node_id in G_lcc.iterNodes():
        community_id = micro_partition[node_id]
        f.write(f"{node_id}\t{community_id}\n")
        count += 1

end_time = time.time()
print(f"-> Successfully wrote {count:,} assignments in {end_time - start_time:.2f} seconds.")




Step 5/6: Writing community assignments to 'output/snapshot_2026-06-26/quantum_comp/louvain_clusters.txt'...
-> Successfully wrote 31,881 assignments in 0.01 seconds.


---
## 🔴 Meso-level

In [22]:
# --- 4. Meso-Level Clustering ---
print("\n---Performing MESO-level clustering...")
start_time = time.time()

import math


def sparsify_local_top_with_msf(G, keep_ratio=0.30):
    """
    Keep local top keep_ratio edges per node (either-end rule), plus
    maximum-spanning-forest safety edges to preserve connectivity structure.
    """
    if not (0.0 < keep_ratio <= 1.0):
        raise ValueError("keep_ratio must be in (0, 1].")
    if G.isDirected():
        raise ValueError("Expected an undirected graph.")

    n = G.numberOfNodes()
    H = nk.Graph(n, weighted=True, directed=False)

    # Build adjacency and edge list.
    nbrs = [[] for _ in range(n)]
    edge_weights = {}
    for u, v in G.iterEdges():
        if u == v:
            continue
        a, b = (u, v) if u < v else (v, u)
        w = float(G.weight(u, v))
        prev = edge_weights.get((a, b))
        if prev is None or w > prev:
            edge_weights[(a, b)] = w

    if not edge_weights:
        return H

    for (a, b), w in edge_weights.items():
        nbrs[a].append((b, w))
        nbrs[b].append((a, w))

    # Either-end local top-k retention.
    local_keep = set()
    for u in range(n):
        deg = len(nbrs[u])
        if deg == 0:
            continue
        k = max(1, int(math.ceil(deg * keep_ratio)))
        top_neighbors = sorted(nbrs[u], key=lambda x: x[1], reverse=True)[:k]
        for v, _ in top_neighbors:
            a, b = (u, v) if u < v else (v, u)
            local_keep.add((a, b))

    # Maximum spanning forest safety edges.
    parent = list(range(n))
    rank = [0] * n

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx == ry:
            return False
        if rank[rx] < rank[ry]:
            parent[rx] = ry
        elif rank[rx] > rank[ry]:
            parent[ry] = rx
        else:
            parent[ry] = rx
            rank[rx] += 1
        return True

    msf_keep = set()
    sorted_edges = sorted(
        ((a, b, w) for (a, b), w in edge_weights.items()),
        key=lambda t: t[2],
        reverse=True,
    )
    for a, b, _ in sorted_edges:
        if union(a, b):
            msf_keep.add((a, b))

    keep_edges = local_keep | msf_keep

    for (a, b), w in edge_weights.items():
        if (a, b) in keep_edges:
            H.addEdge(a, b, w)

    return H


# Create weighted meso graph from micro partition.
G_meso_raw = nk.community.communityGraph(G_lcc, micro_partition)

# Local top-30% sparsification + spanning-forest safety.
G_meso = sparsify_local_top_with_msf(G_meso_raw, keep_ratio=0.30)

print(f"-> Meso raw graph: {G_meso_raw.numberOfNodes():,} nodes, {G_meso_raw.numberOfEdges():,} edges")
print(f"-> Meso pruned graph: {G_meso.numberOfNodes():,} nodes, {G_meso.numberOfEdges():,} edges")

# Parallel Leiden on pruned meso graph.
meso_leiden = nk.community.ParallelLeiden(
    G_meso,
    randomize=True,
    iterations=5,
    gamma=5,
)
meso_leiden.run()
meso_partition = meso_leiden.getPartition()

# Keep meso labels dense for robust meso->macro mapping.
meso_partition.compact()

end_time = time.time()
print(f"-> MESO-level clustering found {meso_partition.numberOfSubsets():,} communities in {end_time - start_time:.2f}s.")
print("-> Meso partition IDs were compacted to a dense range [0..K-1].")


---Performing MESO-level clustering...
-> Meso raw graph: 157 nodes, 10,618 edges
-> Meso pruned graph: 157 nodes, 3,988 edges
-> MESO-level clustering found 48 communities in 0.06s.
-> Meso partition IDs were compacted to a dense range [0..K-1].


---
## 🔴 Macro - level

In [24]:
# --- 5. Macro-Level Clustering ---
print("\n---Performing MACRO-level clustering...")
start_time = time.time()

# Create a final graph where each node is a meso-cluster.
G_macro_raw = nk.community.communityGraph(G_meso, meso_partition)

# Apply the same local top-30% sparsification + spanning-forest safety.
G_macro = sparsify_local_top_with_msf(G_macro_raw, keep_ratio=0.30)

print(f"-> Macro raw graph: {G_macro_raw.numberOfNodes():,} nodes, {G_macro_raw.numberOfEdges():,} edges")
print(f"-> Macro pruned graph: {G_macro.numberOfNodes():,} nodes, {G_macro.numberOfEdges():,} edges")

# Parallel Leiden on pruned macro graph.
macro_leiden = nk.community.ParallelLeiden(
    G_macro,
    randomize=True,
    iterations=5,
    gamma=2,
)
macro_leiden.run()
macro_partition = macro_leiden.getPartition()

# Keep macro labels dense for stable downstream use.
macro_partition.compact()

end_time = time.time()
print(f"-> MACRO-level clustering found {macro_partition.numberOfSubsets():,} communities in {end_time - start_time:.2f}s.")
print("-> Macro partition IDs were compacted to a dense range [0..K-1].")


---Performing MACRO-level clustering...
-> Macro raw graph: 48 nodes, 924 edges
-> Macro pruned graph: 48 nodes, 366 edges
-> MACRO-level clustering found 10 communities in 0.03s.
-> Macro partition IDs were compacted to a dense range [0..K-1].


---
## Write results

In [25]:
# --- 6. Map and Write Results (robust hierarchy checks) ---
# This version is fail-fast against ID-space mismatches across levels.
# It relies on compacted partition IDs and validates that communityGraph node IDs
# align with those compacted IDs before any merge/write step.

import numpy as np

print("\n--- Mapping hierarchical assignments (robust) ---")
start_time = time.time()

# -- Step A: node IDs present in the LCC (preserves original node IDs).
node_ids = np.fromiter(G_lcc.iterNodes(), dtype=np.int64, count=G_lcc.numberOfNodes())

# -- Step B: partition vectors (index = graph node id, value = assigned cluster id).
#    uint64 is required because nk.none = 2^64 - 1.
micro_vec = np.array(micro_partition.getVector(), dtype=np.uint64)
meso_vec  = np.array(meso_partition.getVector(), dtype=np.uint64)
macro_vec = np.array(macro_partition.getVector(), dtype=np.uint64)
none_sentinel = np.uint64(nk.none)

# Ensure every article node got a micro assignment.
micro_assignments = micro_vec[node_ids]
if np.any(micro_assignments == none_sentinel):
    bad = int(np.sum(micro_assignments == none_sentinel))
    raise RuntimeError(f"Found {bad:,} nodes without micro assignment (nk.none).")

# -- Step C: graph-node domains at higher levels.
meso_node_ids  = np.fromiter(G_meso.iterNodes(), dtype=np.int64, count=G_meso.numberOfNodes())
macro_node_ids = np.fromiter(G_macro.iterNodes(), dtype=np.int64, count=G_macro.numberOfNodes())

# Validate expected node-ID domains before joining levels.
expected_micro_ids = np.arange(micro_partition.numberOfSubsets(), dtype=np.int64)
if not np.array_equal(np.sort(meso_node_ids), expected_micro_ids):
    raise RuntimeError(
        "micro->meso ID mismatch: G_meso nodes do not match compact micro IDs. "
        "Aborting to avoid silent misassignment."
    )

expected_meso_ids = np.arange(meso_partition.numberOfSubsets(), dtype=np.int64)
if not np.array_equal(np.sort(macro_node_ids), expected_meso_ids):
    raise RuntimeError(
        "meso->macro ID mismatch: G_macro nodes do not match compact meso IDs. "
        "Aborting to avoid silent misassignment."
    )

# -- Step D: build mapping tables (safe because domains were validated).
meso_map = pd.DataFrame({
    "micro_cluster": meso_node_ids.astype(np.int64),
    "meso_cluster": meso_vec[meso_node_ids].astype(np.int64),
})

macro_map = pd.DataFrame({
    "meso_cluster": macro_node_ids.astype(np.int64),
    "macro_cluster": macro_vec[macro_node_ids].astype(np.int64),
})

# -- Step E: map node -> micro -> meso -> macro.
df = pd.DataFrame({
    "node_id": node_ids,
    "micro_cluster": micro_assignments.astype(np.int64),
})

df = (
    df
    .merge(meso_map, on="micro_cluster", how="left")
    .merge(macro_map, on="meso_cluster", how="left")
)[["node_id", "micro_cluster", "meso_cluster", "macro_cluster"]]

# Final guard: any null here indicates broken cross-level linkage.
null_meso = int(df["meso_cluster"].isna().sum())
null_macro = int(df["macro_cluster"].isna().sum())
if null_meso > 0 or null_macro > 0:
    raise RuntimeError(
        f"Hierarchy mapping produced nulls: meso={null_meso:,}, macro={null_macro:,}."
    )

end_time = time.time()
print(f"-> Mapping complete in {end_time - start_time:.2f} seconds.")
print(f"-> Null checks passed: meso={null_meso:,}, macro={null_macro:,}")

# -- Step F: write to file.
print(f"\nWriting results to '{output_file}' ...")
start_time = time.time()
df.to_csv(output_file, sep="\t", header=True, index=False)
end_time = time.time()
print(f"-> Successfully wrote {len(df):,} assignments in {end_time - start_time:.2f} seconds.")
print("\nProcess complete. ✨")


--- Mapping hierarchical assignments (robust) ---
-> Mapping complete in 0.01 seconds.
-> Null checks passed: meso=0, macro=0

Writing results to 'output/snapshot_2026-06-26/quantum_comp/louvain_clusters.txt' ...
-> Successfully wrote 31,881 assignments in 0.02 seconds.

Process complete. ✨


In [17]:
# --- 7. Post-write hierarchy QC summary ---
# Run this after the mapping/write cell to validate hierarchy integrity quickly.

import pandas as pd

if "df" not in globals():
    raise RuntimeError("DataFrame 'df' is not available. Run the mapping cell first.")

qc = df[["node_id", "micro_cluster", "meso_cluster", "macro_cluster"]].copy()

docs_total = len(qc)
null_meso = int(qc["meso_cluster"].isna().sum())
null_macro = int(qc["macro_cluster"].isna().sum())

micro_to_meso = qc.groupby("micro_cluster")["meso_cluster"].nunique(dropna=True)
micro_to_macro = qc.groupby("micro_cluster")["macro_cluster"].nunique(dropna=True)
meso_to_macro = qc.groupby("meso_cluster")["macro_cluster"].nunique(dropna=True)

micro_without_meso = int((micro_to_meso == 0).sum())
micro_without_macro = int((micro_to_macro == 0).sum())
micro_with_multi_meso = int((micro_to_meso > 1).sum())
micro_with_multi_macro = int((micro_to_macro > 1).sum())
meso_without_macro = int((meso_to_macro == 0).sum())
meso_with_multi_macro = int((meso_to_macro > 1).sum())

summary = pd.DataFrame([
    {
        "docs_total": docs_total,
        "null_meso_docs": null_meso,
        "null_macro_docs": null_macro,
        "micro_clusters": int(qc["micro_cluster"].nunique()),
        "meso_clusters": int(qc["meso_cluster"].nunique()),
        "macro_clusters": int(qc["macro_cluster"].nunique()),
        "micro_without_meso": micro_without_meso,
        "micro_without_macro": micro_without_macro,
        "micro_with_multiple_meso": micro_with_multi_meso,
        "micro_with_multiple_macro": micro_with_multi_macro,
        "meso_without_macro": meso_without_macro,
        "meso_with_multiple_macro": meso_with_multi_macro,
    }
])

print("\n--- Hierarchy QC Summary ---")
print(summary.to_string(index=False))

if micro_with_multi_meso or micro_with_multi_macro or meso_with_multi_macro or null_meso or null_macro:
    print("\n[warn] Potential hierarchy issues detected. Showing top problematic IDs:")

    bad_micro = (
        qc.groupby("micro_cluster", as_index=False)
        .agg(
            docs=("node_id", "count"),
            meso_distinct=("meso_cluster", lambda s: s.nunique(dropna=True)),
            macro_distinct=("macro_cluster", lambda s: s.nunique(dropna=True)),
            null_meso_docs=("meso_cluster", lambda s: int(s.isna().sum())),
            null_macro_docs=("macro_cluster", lambda s: int(s.isna().sum())),
        )
    )
    bad_micro = bad_micro[
        (bad_micro["meso_distinct"] > 1)
        | (bad_micro["macro_distinct"] > 1)
        | (bad_micro["null_meso_docs"] > 0)
        | (bad_micro["null_macro_docs"] > 0)
    ].sort_values(["docs", "micro_cluster"], ascending=[False, True])
    if not bad_micro.empty:
        print("\n[bad micro] top 20")
        print(bad_micro.head(20).to_string(index=False))

    bad_meso = (
        qc.groupby("meso_cluster", as_index=False)
        .agg(
            docs=("node_id", "count"),
            macro_distinct=("macro_cluster", lambda s: s.nunique(dropna=True)),
            null_macro_docs=("macro_cluster", lambda s: int(s.isna().sum())),
        )
    )
    bad_meso = bad_meso[
        (bad_meso["macro_distinct"] > 1)
        | (bad_meso["null_macro_docs"] > 0)
    ].sort_values(["docs", "meso_cluster"], ascending=[False, True])
    if not bad_meso.empty:
        print("\n[bad meso] top 20")
        print(bad_meso.head(20).to_string(index=False))
else:
    print("\n[ok] Hierarchy mapping checks passed (no nulls and one-to-one parent links).")


--- Hierarchy QC Summary ---
 docs_total  null_meso_docs  null_macro_docs  micro_clusters  meso_clusters  macro_clusters  micro_without_meso  micro_without_macro  micro_with_multiple_meso  micro_with_multiple_macro  meso_without_macro  meso_with_multiple_macro
      31881               0                0             157              2               1                   0                    0                         0                          0                   0                         0

[ok] Hierarchy mapping checks passed (no nulls and one-to-one parent links).
